<table>
<tr>
<td>

# TFG - Entrenamiento de Modelos de Predicción

### Estimación de rendimiento de cultivos utilizando inteligencia artificial y datos de observación de la tierra.

**Nacor Olmos Caballero**

</td>
<td>
<img src="./img/logo_uv.png" width="250" height="250">
</td>
</tr>
</table>

## Librerías

In [1]:
import pandas as pd
import numpy as np
import pprint
import joblib


from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import os
os.makedirs("Resultados", exist_ok=True)
os.makedirs("Mejores_modelos", exist_ok=True)

## Datos

In [4]:
maiz = pd.read_csv("./Clean_Data/df_maize_final.csv", index_col=0)
arroz = pd.read_csv("./Clean_Data/df_rice_final.csv", index_col=0)
trigo = pd.read_csv("./Clean_Data/df_wheat_final.csv", index_col=0)

maiz.Name = "maiz"
arroz.Name = "arroz"
trigo.Name = "trigo"

# DataFrames para nombres de columnas
col_names_train_maize = pd.read_csv("./Clean_Data/mod_maiz.csv", index_col=0).columns.tolist()
col_names_train_rice = pd.read_csv("./Clean_Data/mod_arroz.csv", index_col=0).columns.tolist()
col_names_train_wheat = pd.read_csv("./Clean_Data/mod_trigo.csv", index_col=0).columns.tolist()

Los datos que tenemos son los siguientes:

· maiz/arroz/trigo -> Dataframes finales de cada cultivo.

· col_names_train_'cultivo' -> Listas que contienen las variables óptimas para el modelado según el análisis exploratorio de los datos.

In [5]:
print("Columnas de entrenamiento para maíz:", col_names_train_maize[:-1])
print("Datos de maíz:\n", maiz.head())

Columnas de entrenamiento para maíz: ['mean_ASAP:RAIN', 'mean_ASAP:SM_combined', 'mean_ASAP:TEMP', 'mean_ASAP:rad', 'mean_ASIS:ASIS_NDVI', 'mean_ASIS:BT4', 'std_ASAP:FPAR', 'std_ASAP:SM_combined', 'std_ASAP:TEMP', 'std_ASAP:rad', 'std_ASIS:ASIS_NDVI', 'std_ASIS:BT4', 'koppen_class']
Datos de maíz:
               year  mean_ASAP:FPAR  mean_ASAP:RAIN  mean_ASAP:SM  \
asap_unit_id                                                       
41            1984       53.888647       54.665893      0.277685   
41            1985       53.888647       48.624003      0.275522   
41            1986       53.888647       49.286758      0.273614   
41            1987       53.888647       48.611285      0.266157   
41            1988       53.888647       46.851524      0.270388   

              mean_ASAP:SM_combined  mean_ASAP:TEMP  mean_ASAP:rad  \
asap_unit_id                                                         
41                         0.282407       23.225820  173414.084765   
41           

## Modelos

### Definición de modelos que usaremos para el entrenamiento

In [6]:
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_xgb = XGBRegressor(n_estimators=100, random_state=42)
model_lgb = LGBMRegressor(n_estimators=100, random_state=42, min_data_in_bin=1, min_data_in_leaf=1, force_col_wise=True, verbose=-1)

### Búsqueda de parámetros óptimos para entrenar los modelos SVR

#### Optimización Bayesiana

In [7]:
# Optimización bayesiana
def opt_bayes_SVR(df_cultivo, col_names_train, random_state=None, n_iter=30):
    """
    Optimización bayesiana para SVR con un DataFrame específico y nombres de columnas.
    
    Args:
        df_cultivo (pd.DataFrame): DataFrame con los datos del cultivo.
        col_names_train (list): Lista de nombres de columnas para el entrenamiento.
        n_iter (int): Número de iteraciones para la optimización bayesiana.
        
    Returns:
        model_svr (SVR): Modelo SVR optimizado.
    """
    x = df_cultivo[col_names_train[:-1] + ['year']]
    y = df_cultivo["yield"]
    x.columns = x.columns.str.replace(':', '_', regex=False)

    pipe = make_pipeline(StandardScaler(), SVR())

    param_space = {
        "svr__C": Real(1e-2, 1e2, prior='log-uniform'),
        # "svr__epsilon": Real(0.0001, 0.1, prior='log-uniform'),
        "svr__epsilon": Real(0.01, 0.2, prior='log-uniform'), # Reentrenamos con Epsilon más alto (con respecto a '1e-4, 1e-1') para evitar sobreajuste del modelo
        "svr__gamma": Real(1e-4, 1e-1, prior='log-uniform'),
        "svr__kernel": Categorical(["rbf", "linear", "poly"])
    }

    tscv = TimeSeriesSplit(n_splits=3)

    opt = BayesSearchCV(
        estimator=pipe,
        search_spaces=param_space,
        n_iter=n_iter,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        random_state=random_state,
        verbose=0
    )

    opt.fit(x, y)
    
    # Extraemos modelo SVR dentro del pipeline
    modelo_svr = opt.best_estimator_.named_steps["svr"]

    # Mostramos los resultados
    print("Número total de vectores soporte:", modelo_svr.n_support_.sum())
    print("Mejores parámetros encontrados:", opt.best_params_)
    print("Mejor puntuación (RMSE):", -opt.best_score_)

    return opt.best_estimator_, opt.best_params_, -opt.best_score_

In [8]:
print("Resultados SVR - Maíz")
svr_bayes_maiz1, svr_bayes_maiz1_params, svr_bayes_maiz1_rmse = opt_bayes_SVR(maiz, col_names_train_maize, random_state=42)
svr_bayes_maiz2, svr_bayes_maiz2_params, svr_bayes_maiz2_rmse = opt_bayes_SVR(maiz, col_names_train_maize, random_state=502)
svr_bayes_maiz3, svr_bayes_maiz3_params, svr_bayes_maiz3_rmse = opt_bayes_SVR(maiz, col_names_train_maize, random_state=6002)
svr_bayes_maiz4, svr_bayes_maiz4_params, svr_bayes_maiz4_rmse = opt_bayes_SVR(maiz, col_names_train_maize, random_state=70002)
svr_bayes_maiz5, svr_bayes_maiz5_params, svr_bayes_maiz5_rmse = opt_bayes_SVR(maiz, col_names_train_maize, random_state=800002)

print("\nResultados SVR - Arroz")
svr_bayes_rice1, svr_bayes_rice1_params, svr_bayes_rice1_rmse = opt_bayes_SVR(arroz, col_names_train_rice, random_state=42)
svr_bayes_rice2, svr_bayes_rice2_params, svr_bayes_rice2_rmse = opt_bayes_SVR(arroz, col_names_train_rice, random_state=502)
svr_bayes_rice3, svr_bayes_rice3_params, svr_bayes_rice3_rmse = opt_bayes_SVR(arroz, col_names_train_rice, random_state=6002)
svr_bayes_rice4, svr_bayes_rice4_params, svr_bayes_rice4_rmse = opt_bayes_SVR(arroz, col_names_train_rice, random_state=70002)
svr_bayes_rice5, svr_bayes_rice5_params, svr_bayes_rice5_rmse = opt_bayes_SVR(arroz, col_names_train_rice, random_state=800002)

print("\nResultados SVR - Trigo")
svr_bayes_wheat1, svr_bayes_wheat1_params, svr_bayes_wheat1_rmse = opt_bayes_SVR(trigo, col_names_train_wheat, random_state=42)
svr_bayes_wheat2, svr_bayes_wheat2_params, svr_bayes_wheat2_rmse = opt_bayes_SVR(trigo, col_names_train_wheat, random_state=502)
svr_bayes_wheat3, svr_bayes_wheat3_params, svr_bayes_wheat3_rmse = opt_bayes_SVR(trigo, col_names_train_wheat, random_state=6002)
svr_bayes_wheat4, svr_bayes_wheat4_params, svr_bayes_wheat4_rmse = opt_bayes_SVR(trigo, col_names_train_wheat, random_state=70002)
svr_bayes_wheat5, svr_bayes_wheat5_params, svr_bayes_wheat5_rmse = opt_bayes_SVR(trigo, col_names_train_wheat, random_state=800002)

Resultados SVR - Maíz
Número total de vectores soporte: 858
Mejores parámetros encontrados: OrderedDict({'svr__C': 11.343959780192792, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'linear'})
Mejor puntuación (RMSE): 1016.1035604719372
Número total de vectores soporte: 858
Mejores parámetros encontrados: OrderedDict({'svr__C': 12.11008468278404, 'svr__epsilon': 0.01, 'svr__gamma': 0.0001, 'svr__kernel': 'linear'})
Mejor puntuación (RMSE): 1012.3089711665322
Número total de vectores soporte: 858
Mejores parámetros encontrados: OrderedDict({'svr__C': 31.175665141279712, 'svr__epsilon': 0.2, 'svr__gamma': 0.1, 'svr__kernel': 'poly'})
Mejor puntuación (RMSE): 1106.5665610135966
Número total de vectores soporte: 858
Mejores parámetros encontrados: OrderedDict({'svr__C': 12.351538893660782, 'svr__epsilon': 0.2, 'svr__gamma': 0.0001, 'svr__kernel': 'linear'})
Mejor puntuación (RMSE): 1012.8997241527258
Número total de vectores soporte: 857
Mejores parámetros encontrados: OrderedDict

c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.01, 0.1, np.str_('poly')] before, using random point [17.58055651616896, 0.053887380700573514, 0.0006013680827208865, 'poly']
  warnings.warn(
c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.01, 0.1, np.str_('poly')] before, using random point [7.885188658300148, 0.12600977384633064, 0.0009375167616889421, 'poly']
  warnings.warn(
c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.01, 0.1, np.str_('poly')] before, using random point [0.11894205178261723, 0.023095438323152643, 0.011497888538324045, 'rbf']
  warnings.warn(
c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarni

Número total de vectores soporte: 630
Mejores parámetros encontrados: OrderedDict({'svr__C': 100.0, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'poly'})
Mejor puntuación (RMSE): 1182.7141290342927
Número total de vectores soporte: 630
Mejores parámetros encontrados: OrderedDict({'svr__C': 0.18654284844488936, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'linear'})
Mejor puntuación (RMSE): 1215.050006770269
Número total de vectores soporte: 630
Mejores parámetros encontrados: OrderedDict({'svr__C': 0.17799373681819503, 'svr__epsilon': 0.12286094629630964, 'svr__gamma': 0.1, 'svr__kernel': 'linear'})
Mejor puntuación (RMSE): 1214.9331736619731
Número total de vectores soporte: 630
Mejores parámetros encontrados: OrderedDict({'svr__C': 58.63545512140447, 'svr__epsilon': 0.1978253445053047, 'svr__gamma': 0.026630704308941773, 'svr__kernel': 'poly'})
Mejor puntuación (RMSE): 1203.9129393962567


c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.2, 0.1, np.str_('poly')] before, using random point [1.6363810168346347, 0.016641400791339474, 0.0016173272040228795, 'linear']
  warnings.warn(
c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.2, 0.1, np.str_('poly')] before, using random point [10.4798947947285, 0.18169989953254995, 0.055784433376176376, 'linear']
  warnings.warn(
c:\Users\Nacor Olmos\anaconda3\envs\TFG\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [100.0, 0.2, 0.1, np.str_('poly')] before, using random point [58.634330455871826, 0.14725113397311637, 0.051954203073443, 'poly']
  warnings.warn(


Número total de vectores soporte: 630
Mejores parámetros encontrados: OrderedDict({'svr__C': 100.0, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'poly'})
Mejor puntuación (RMSE): 1182.7141290342927


In [9]:
resultados_bayes_bayes = pd.DataFrame([
    {"cultivo": "maiz", "modelo": "svr_bayes_maiz1", "params": svr_bayes_maiz1_params, "RMSE": svr_bayes_maiz1_rmse},
    {"cultivo": "maiz", "modelo": "svr_bayes_maiz2", "params": svr_bayes_maiz2_params, "RMSE": svr_bayes_maiz2_rmse},
    {"cultivo": "maiz", "modelo": "svr_bayes_maiz3", "params": svr_bayes_maiz3_params, "RMSE": svr_bayes_maiz3_rmse},
    {"cultivo": "maiz", "modelo": "svr_bayes_maiz4", "params": svr_bayes_maiz4_params, "RMSE": svr_bayes_maiz4_rmse},
    {"cultivo": "maiz", "modelo": "svr_bayes_maiz5", "params": svr_bayes_maiz5_params, "RMSE": svr_bayes_maiz5_rmse},
    {"cultivo": "arroz", "modelo": "svr_bayes_rice1", "params": svr_bayes_rice1_params, "RMSE": svr_bayes_rice1_rmse},
    {"cultivo": "arroz", "modelo": "svr_bayes_rice2", "params": svr_bayes_rice2_params, "RMSE": svr_bayes_rice2_rmse},
    {"cultivo": "arroz", "modelo": "svr_bayes_rice3", "params": svr_bayes_rice3_params, "RMSE": svr_bayes_rice3_rmse},
    {"cultivo": "arroz", "modelo": "svr_bayes_rice4", "params": svr_bayes_rice4_params, "RMSE": svr_bayes_rice4_rmse},
    {"cultivo": "arroz", "modelo": "svr_bayes_rice5", "params": svr_bayes_rice5_params, "RMSE": svr_bayes_rice5_rmse},
    {"cultivo": "trigo", "modelo": "svr_bayes_wheat1", "params": svr_bayes_wheat1_params, "RMSE": svr_bayes_wheat1_rmse},
    {"cultivo": "trigo", "modelo": "svr_bayes_wheat2", "params": svr_bayes_wheat2_params, "RMSE": svr_bayes_wheat2_rmse},
    {"cultivo": "trigo", "modelo": "svr_bayes_wheat3", "params": svr_bayes_wheat3_params, "RMSE": svr_bayes_wheat3_rmse},
    {"cultivo": "trigo", "modelo": "svr_bayes_wheat4", "params": svr_bayes_wheat4_params, "RMSE": svr_bayes_wheat4_rmse},
    {"cultivo": "trigo", "modelo": "svr_bayes_wheat5", "params": svr_bayes_wheat5_params, "RMSE": svr_bayes_wheat5_rmse}
])

resultados_ordenados = resultados_bayes_bayes.sort_values(by=["cultivo", "RMSE"], ascending=[True, True])

for i, row in resultados_ordenados.iterrows():
    print(f"{row['cultivo'].upper()} | {row['modelo']} | RMSE: {row['RMSE']:.2f}")
    pprint.pprint(row['params'])
    print("-" * 60)

ARROZ | svr_bayes_rice4 | RMSE: 1441.39
OrderedDict([('svr__C', 26.038110572393766),
             ('svr__epsilon', 0.01),
             ('svr__gamma', 0.048877663922770605),
             ('svr__kernel', 'rbf')])
------------------------------------------------------------
ARROZ | svr_bayes_rice1 | RMSE: 1443.62
OrderedDict([('svr__C', 22.674130002881192),
             ('svr__epsilon', 0.08692247599577088),
             ('svr__gamma', 0.1),
             ('svr__kernel', 'rbf')])
------------------------------------------------------------
ARROZ | svr_bayes_rice5 | RMSE: 1451.75
OrderedDict([('svr__C', 11.864007572547997),
             ('svr__epsilon', 0.16872795224874915),
             ('svr__gamma', 0.08504534514699702),
             ('svr__kernel', 'rbf')])
------------------------------------------------------------
ARROZ | svr_bayes_rice3 | RMSE: 1451.98
OrderedDict([('svr__C', 10.173716873936845),
             ('svr__epsilon', 0.2),
             ('svr__gamma', 0.1),
             ('sv

#### GridSearchCV

In [10]:
def opt_grid_SVR(df_cultivo,col_names_train):
    """
    Optimización por Grid Search CV para SVR.

    Args:
        df_cultivo (pd.DataFrame): DataFrame con los datos del cultivo.
        col_names_train (list): Lista de nombres de columnas para el entrenamiento.

    Returns:
        model_svr (SVR): Modelo SVR optimizado.
    """
    x = df_cultivo[col_names_train[:-1] + ['year']]
    y = df_cultivo["yield"]
    x.columns = x.columns.str.replace(':', '_', regex=False)

    pipe = make_pipeline(StandardScaler(), SVR())

    param_grid = {
        "svr__C": [1e-2, 1e-1, 1e0, 1e1, 1e2],
        # "svr__epsilon": [0.0001, 0.001, 0.01, 0.1], 
        "svr__epsilon": [0.01, 0.1, 0.2], # Reentrenamos con Epsilon más alto (con respecto a '1e-4, 1e-1') para evitar sobreajuste del modelo
        "svr__gamma": [1e-4, 1e-3, 1e-2, 1e-1],
        "svr__kernel": ["rbf", "linear", "poly"]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=TimeSeriesSplit(n_splits=3),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        verbose=0
    )

    grid.fit(x, y)

    # Extraemos modelo SVR dentro del pipeline
    modelo_svr = grid.best_estimator_.named_steps["svr"]

    # Mostramos los resultados
    print("Número total de vectores soporte:", modelo_svr.n_support_.sum())
    print(f"Mejores parámetros encontrados para {df_cultivo.Name}:", grid.best_params_)
    print("Mejor puntuación (RMSE):", -grid.best_score_)

    return grid.best_estimator_

In [11]:
svr_grid_maiz = opt_grid_SVR(maiz, col_names_train_maize)
svr_grid_arroz = opt_grid_SVR(arroz, col_names_train_rice)
svr_grid_trigo = opt_grid_SVR(trigo, col_names_train_wheat)

Número total de vectores soporte: 858
Mejores parámetros encontrados para maiz: {'svr__C': 10.0, 'svr__epsilon': 0.01, 'svr__gamma': 0.0001, 'svr__kernel': 'linear'}
Mejor puntuación (RMSE): 1019.8838145147
Número total de vectores soporte: 702
Mejores parámetros encontrados para arroz: {'svr__C': 10.0, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'rbf'}
Mejor puntuación (RMSE): 1451.961678506653
Número total de vectores soporte: 630
Mejores parámetros encontrados para trigo: {'svr__C': 100.0, 'svr__epsilon': 0.01, 'svr__gamma': 0.1, 'svr__kernel': 'poly'}
Mejor puntuación (RMSE): 1182.7141290342927


#### Conclusiones y elección del modelo final

Observamos que todos los modelos entrenados (Opt. Bay. & GridSearchCV) han empleado todos o la mayoría de los datos como vectores soporte lo que nos indica que los modelos sobreajustan, aún habiendo aumentado los valores de 'Epsilon' para tener un mayor margen de tolerancia.

##### - Optimización Bayesiana:

· Maíz -> Vemos que hay tres iteraciones con resultados similares, que emplean valores de 'C', 'epsilon' y 'gamma' similares, pero comparten kernel tipo 'linear'; las otras dos iteraciones tienen resultados visualmente peores, con valores similares en 'epsilon' y 'gamma'.

Nos quedaremos con la configuración de *svr_bayes_maiz2 -> [('RMSE', 1012.31), ('svr__C', 12.11008468278404), ('svr__epsilon', 0.01), ('svr__gamma', 0.0001), ('svr__kernel', 'linear')]*.

· Arroz -> Para este cultivo observamos lo siguiente: Predomina el tipo de kernel 'rbf' y disminuir los valores de 'epsilon' y 'gamma', a la vez que se aumenta el de 'C', influye de forma positiva en el RMSE (Podríamos probar manualmente si esta tendencia se cumple).

Nos quedaremos con la configuración de *svr_bayes_rice4 -> [('RMSE', 1441.39), ('svr__C', 26.038110572393766), ('svr__epsilon', 0.01), ('svr__gamma', 0.048877663922770605), ('svr__kernel', 'rbf')]*.

· Trigo ->  En este caso hay dos modelos con parámetros idénticos, lo que nos indica que podemos haber encontrado un óptimo global para este cultivo. Antes, los valores de los 'random_state' estaban definidos más próximos (42, 52, 62, 72, 82, respectivamente) y para Trigo las 5 iteraciones dieron este mismo resultado.

Nos quedaremos con la configuración de *svr_bayes_wheat1 -> [('RMSE', 1182.71), ('svr__C', 100.0), ('svr__epsilon', 0.01), ('svr__gamma', 0.1), ('svr__kernel', 'poly')]*.


##### - GridSearchCV:

Observamos resultados peores en los cultivos de Maíz y Arroz e idénticos en Trigo (refuerza la idea de óptimo global), con respecto a los obtenidos con Optimización Bayesiana.


##### - Conclusión final:

Hemos visto que la Optimización Bayesiana se ajusta en mayor medida a nuestro caso de estudio ya que los tiempos de ejecución no son muy elevados (apenas 6 min.) y obtiene mejores resultados que GridSearchCV.

Para entrenar los modelos usaremos las configuraciones siguientes:

· Maíz -> *svr_bayes_maiz2*, el kernel 'lineal' ofrece un rendimiento razonablemente estable para maíz. El modelo necesita una penalización moderada ('C' ≈ 12) y un margen de error (tolerancia) controlado. 

· Arroz -> *svr_bayes_rice4*, el kernel 'rbf' ofece buena flexibilidad, una penalización alta ('C' ≈ 26) y un margen de error pequeño obtienen buenos resultados.

· Trigo -> *svr_bayes_wheat1*, encuentra un óptimo robusto usando kernel polinómico ('poly'), pero el valor alto de 'C' (=100) y el uso de todos los vectores soporte sugieren una fuerte dependencia del conjunto de entrenamiento y, por tanto, un alto riesgo de sobreajuste.

In [12]:
params_svr_maiz_final = svr_bayes_maiz2
params_svr_arroz_final = svr_bayes_rice4
params_svr_trigo_final = svr_bayes_wheat1

### Estrategias de entrenamiento

#### · K-Fold por años:

Excluimos un año en entrenamiento y lo usaremos para validación, repetimos para cada año.

#### · Ventana temporal:

Estudiamos un año en base a los anteriores.

### Metodologías:



#### 1. Entrenamiento de modelos agrupando solo por país (Al haber pocos datos puede conducirnos a sobreentrenamiento)

Procedimiento: Analizar, para cada cultivo, los datos de cada país por separado.

##### K-Fold

In [13]:
def kfold_cult_pais(df_cultivo, col_names_cultivo, model):
    """
    Entrena un modelo para cada país en el DataFrame dado usando K-Fold por año.

    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados de cada país.
    """

    resultados = []

    # Lista de países
    paises = df_cultivo['country'].unique().tolist()
    
    for pais in paises:
        # print(f"Entrenando modelo para {pais}...")

        df_pais = df_cultivo[df_cultivo["country"] == pais]

        anyos = df_pais["year"].unique().tolist()

        if len(anyos) < 5:
            print(f"Menos de 5 años de datos para {pais}, saltando...")
            continue

        for anyo in anyos:
            df_train = df_pais[df_pais["year"] != anyo] # Entrenamiento con todos los años excepto el actual
            df_test = df_pais[df_pais["year"] == anyo] # Test con el año actual

            koppen = df_test["koppen_class"].iloc[0]
            
            # División de datos de entrenamiento y validación
            X_train = df_train[col_names_cultivo[:-1] + ["year"]]
            y_train = df_train["yield"]
            X_test = df_test[col_names_cultivo[:-1] + ["year"]]
            y_test = df_test["yield"]

            X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
            X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

            # Entrenamiento del modelo
            model_current = clone(model)
            model_current.fit(X_train, y_train)

            # Predecimos
            y_pred = model_current.predict(X_test)

            # Evaluamos las predicciones del modelo
            # r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
            mae = mean_absolute_error(y_test, y_pred) # Error absoluto medio
            me = np.sum(y_test - y_pred) / len(y_test) # Error medio

            # Normalizamos los resultados
            nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
            rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
            mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
            nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

            resultados.append({
                "country": pais,
                "year": anyo,
                "y_true": y_test.values.tolist(),
                "y_pred": y_pred.tolist(),
                # "R2": r2 if r2 is not None else 0,
                "NRMSE": nrmse,
                "rRMSE": rRMSE,
                "NMAE": mae,
                "nME": nme,
                "koppen_class": koppen,
            })

    df_resultados = pd.DataFrame(resultados)

    # Entrenamos un modelo final sobre todos los datos
    X_total = df_cultivo[col_names_cultivo[:-1] + ["year"]]
    y_total = df_cultivo["yield"]
    X_total.columns = X_total.columns.str.replace(":", "_", regex=False)

    model_final = clone(model)
    model_final.fit(X_total, y_total)


    return df_resultados, model_final

In [14]:
print("Entrenamos Random Forest con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_rf, modelo_rf_kfold_maiz1  = kfold_cult_pais(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_kfold_arroz_rf, modelo_rf_kfold_arroz1  = kfold_cult_pais(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_kfold_trigo_rf, modelo_rf_kfold_trigo1  = kfold_cult_pais(trigo, col_names_train_wheat, model_rf)
print("Modelos entrenados.")

Entrenamos Random Forest con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [15]:
print("Entrenamos XGBoost con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_xgb, modelo_xgb_kfold_maiz1 = kfold_cult_pais(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_xgb, modelo_xgb_kfold_arroz1 = kfold_cult_pais(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_xgb, modelo_xgb_kfold_trigo1 = kfold_cult_pais(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [16]:
print("Entrenamos LightGBM con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_lgb, modelo_lgb_kfold_maiz1 = kfold_cult_pais(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_lgb, modelo_lgb_kfold_arroz1 = kfold_cult_pais(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_lgb, modelo_lgb_kfold_trigo1 = kfold_cult_pais(trigo, col_names_train_wheat, model_lgb)
print("Modelos entrenados.")

Entrenamos LightGBM con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [17]:
print("Entrenamos SVR con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_svr, aux  = kfold_cult_pais(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_kfold_arroz_svr, aux = kfold_cult_pais(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_kfold_trigo_svr, aux = kfold_cult_pais(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [18]:
df_resultados_kfold_maiz_rf.to_csv("./Resultados/resultados_kfold_maiz_rf1.csv", index=False)
joblib.dump(modelo_rf_kfold_maiz1, "./Mejores_modelos/training_kfold_maiz_rf1.pkl")
df_resultados_kfold_arroz_rf.to_csv("./Resultados/resultados_kfold_arroz_rf1.csv", index=False)
joblib.dump(modelo_rf_kfold_arroz1, "./Mejores_modelos/training_kfold_arroz_rf1.pkl")
df_resultados_kfold_trigo_rf.to_csv("./Resultados/resultados_kfold_trigo_rf1.csv", index=False)
joblib.dump(modelo_rf_kfold_trigo1, "./Mejores_modelos/training_kfold_trigo_rf1.pkl")

df_resultados_kfold_maiz_xgb.to_csv("./Resultados/resultados_kfold_maiz_xgb1.csv", index=False)
joblib.dump(modelo_xgb_kfold_maiz1, "./Mejores_modelos/training_kfold_maiz_xgb1.pkl")
df_resultados_kfold_arroz_xgb.to_csv("./Resultados/resultados_kfold_arroz_xgb1.csv", index=False)
joblib.dump(modelo_xgb_kfold_arroz1, "./Mejores_modelos/training_kfold_arroz_xgb1.pkl")
df_resultados_kfold_trigo_xgb.to_csv("./Resultados/resultados_kfold_trigo_xgb1.csv", index=False)
joblib.dump(modelo_xgb_kfold_trigo1, "./Mejores_modelos/training_kfold_trigo_xgb1.pkl")

df_resultados_kfold_maiz_lgb.to_csv("./Resultados/resultados_kfold_maiz_lgb1.csv", index=False)
joblib.dump(modelo_lgb_kfold_maiz1, "./Mejores_modelos/training_kfold_maiz_lgb1.pkl")
df_resultados_kfold_arroz_lgb.to_csv("./Resultados/resultados_kfold_arroz_lgb1.csv", index=False)
joblib.dump(modelo_lgb_kfold_arroz1, "./Mejores_modelos/training_kfold_arroz_lgb1.pkl")
df_resultados_kfold_trigo_lgb.to_csv("./Resultados/resultados_kfold_trigo_lgb1.csv", index=False)
joblib.dump(modelo_lgb_kfold_trigo1, "./Mejores_modelos/training_kfold_trigo_lgb1.pkl")

df_resultados_kfold_maiz_svr.to_csv("./Resultados/resultados_kfold_maiz_svr1.csv", index=False)
df_resultados_kfold_arroz_svr.to_csv("./Resultados/resultados_kfold_arroz_svr1.csv", index=False)
df_resultados_kfold_trigo_svr.to_csv("./Resultados/resultados_kfold_trigo_svr1.csv", index=False)

##### Ventana temporal

In [19]:
def vent_temp_cult_pais(df_cultivo, col_names_cultivo, model):
    """
    Entrena un modelo para cada país en el DataFrame dado usando una ventana temporal.

    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados de cada país.
    """

    resultados = []

    # Lista de países
    paises = df_cultivo['country'].unique().tolist()

    for pais in paises:
        df_pais = df_cultivo[df_cultivo["country"] == pais]
        df_pais = df_pais.sort_values(by="year")
        anyos = df_pais["year"].unique().tolist()

        if len(anyos) < 5:
            print(f"Menos de 5 años de datos para {pais}, saltando...")
            continue

        for i in range(3, len(anyos)):
            anyo = anyos[i]
            anyos_ant = anyos[:i] # Todos los años anteriores al actual

            df_train = df_pais[df_pais["year"].isin(anyos_ant)] # Entrenamiento con todos los años anteriores al actual
            df_test = df_pais[df_pais["year"] == anyo] # Test con el año actual

            koppen = df_test["koppen_class"].iloc[0]

            X_train = df_train[col_names_cultivo[:-1] + ["year"]]
            y_train = df_train["yield"]
            X_test = df_test[col_names_cultivo[:-1] + ["year"]]
            y_test = df_test["yield"]

            X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
            X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

            if len(X_train) == 0 or len(X_test) == 0:
                continue

            # Entrenamiento del modelo
            model_current = clone(model)
            model_current.fit(X_train, y_train)

            # Predecimos
            y_pred = model_current.predict(X_test)

            # Evaluamos las predicciones del modelo
            # r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae = mean_absolute_error(y_test, y_pred)
            me = np.sum(y_test - y_pred) / len(y_test) # Error medio

            # Normalizamos los resultados
            nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
            rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
            mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
            nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

            resultados.append({
                "country": pais,
                "year": anyo,
                "y_true": y_test.values.tolist(),
                "y_pred": y_pred.tolist(),
                # "R2": r2,
                "NRMSE": nrmse,
                "rRMSE": rRMSE,
                "NMAE": mae,
                "nME": nme,
                "koppen_class": koppen,
            })

    df_resultados = pd.DataFrame(resultados)

    # Entrenamos un modelo final sobre todos los datos
    X_total = df_cultivo[col_names_cultivo[:-1] + ["year"]]
    y_total = df_cultivo["yield"]
    X_total.columns = X_total.columns.str.replace(":", "_", regex=False)

    model_final = clone(model)
    model_final.fit(X_total, y_total)


    return df_resultados, model_final

In [20]:
print("Entrenamos Random Forest con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_rf, aux = vent_temp_cult_pais(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_rf, aux = vent_temp_cult_pais(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_rf, aux = vent_temp_cult_pais(trigo, col_names_train_wheat, model_rf)
print("Modelos entrenados.")

Entrenamos Random Forest con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [21]:
print("Entrenamos XGBoost con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_xgb, aux = vent_temp_cult_pais(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_xgb, aux = vent_temp_cult_pais(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_xgb, aux = vent_temp_cult_pais(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [22]:
print("Entrenamos LightGBM con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_lgb, modelo_svr_vent_maiz1 = vent_temp_cult_pais(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_lgb, modelo_svr_vent_arroz1 = vent_temp_cult_pais(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_lgb, aux = vent_temp_cult_pais(trigo, col_names_train_wheat, model_lgb)
print("Modelos entrenados.")

Entrenamos LightGBM con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [23]:
print("Entrenamos SVR con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_svr, aux = vent_temp_cult_pais(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_svr, aux = vent_temp_cult_pais(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_svr, aux = vent_temp_cult_pais(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [24]:
df_resultados_vent_temp_maiz_rf.to_csv("./Resultados/resultados_vent_temp_maiz_rf1.csv", index=False)
df_resultados_vent_temp_arroz_rf.to_csv("./Resultados/resultados_vent_temp_arroz_rf1.csv", index=False)
df_resultados_vent_temp_trigo_rf.to_csv("./Resultados/resultados_vent_temp_trigo_rf1.csv", index=False)

df_resultados_vent_temp_maiz_xgb.to_csv("./Resultados/resultados_vent_temp_maiz_xgb1.csv", index=False)
df_resultados_vent_temp_arroz_xgb.to_csv("./Resultados/resultados_vent_temp_arroz_xgb1.csv", index=False)
df_resultados_vent_temp_trigo_xgb.to_csv("./Resultados/resultados_vent_temp_trigo_xgb1.csv", index=False)

df_resultados_vent_temp_maiz_lgb.to_csv("./Resultados/resultados_vent_temp_maiz_lgb1.csv", index=False)
df_resultados_vent_temp_arroz_lgb.to_csv("./Resultados/resultados_vent_temp_arroz_lgb1.csv", index=False)
df_resultados_vent_temp_trigo_lgb.to_csv("./Resultados/resultados_vent_temp_trigo_lgb1.csv", index=False)

df_resultados_vent_temp_maiz_svr.to_csv("./Resultados/resultados_vent_temp_maiz_svr1.csv", index=False)
joblib.dump(modelo_svr_vent_maiz1, "./Mejores_modelos/training_vent_maiz_svr1.pkl")
df_resultados_vent_temp_arroz_svr.to_csv("./Resultados/resultados_vent_temp_arroz_svr1.csv", index=False)
joblib.dump(modelo_svr_vent_arroz1, "./Mejores_modelos/training_vent_arroz_svr1.pkl")
df_resultados_vent_temp_trigo_svr.to_csv("./Resultados/resultados_vent_temp_trigo_svr1.csv", index=False)

#### 2. Entrenamiento de modelos agrupando paises por clase koppen (tipo de clima) y tipo de cultivo 

Procedimiento: Analizar, para cada cultivo, los datos de cada clase koppen por separado.

##### K-Fold

In [25]:
def kfold_cult_koppen(df_cultivo, col_names_cultivo, model):
    """
    Crea un DataFrame con los resultados de K-Fold por cultivo y clase de Köppen.
    
    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados de K-Fold por cultivo y clase de Köppen.
    """
    resultados = []

    # Lista de climas
    climas = df_cultivo['koppen_class'].unique().tolist()

    for clima in climas:
        df_clima = df_cultivo[df_cultivo["koppen_class"] == clima]
        anyos = df_clima["year"].unique().tolist()

        if len(anyos) < 5:
            print(f"Menos de 5 años de datos para {clima}, saltando...")
            continue

        for anyo in anyos:
            df_train = df_clima[df_clima["year"] != anyo] # Entrenamiento con todos los años excepto el actual
            df_test = df_clima[df_clima["year"] == anyo] # Test con el año actual

            X_train = df_train[col_names_cultivo[:-1] + ["year"]]
            y_train = df_train["yield"]
            X_test = df_test[col_names_cultivo[:-1] + ["year"]]
            y_test = df_test["yield"]

            X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
            X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

            # # Verificamos si hay suficientes datos para entrenar
            # if len(X_train) < 10 or len(X_test) < 10:
            #     print(f"Datos insuficientes para entrenar el modelo para {clima} en el año {anyo}, saltando...")
            #     continue

            # Entrenamiento del modelo
            model_current = clone(model)
            model_current.fit(X_train, y_train)

            # Predecimos
            y_pred = model_current.predict(X_test)

            # Evaluamos las predicciones del modelo
            # r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
            mae = mean_absolute_error(y_test, y_pred) # Error absoluto medio
            me = np.sum(y_test - y_pred) / len(y_test) # Error medio

            # Normalizamos los resultados
            nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
            rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
            mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
            nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

            resultados.append({
                "koppen_class": clima,
                "year": anyo,
                "y_true": y_test.values.tolist(),
                "y_pred": y_pred.tolist(),
                # "R2": r2 if r2 is not None else 0,
                "NRMSE": nrmse,
                "rRMSE": rRMSE,
                "NMAE": mae,
                "nME": nme,
            })

    return pd.DataFrame(resultados)

In [26]:
print("Entrenamos Random Forest con K-Fold por clase de Köppen...")
print("Entrenando maíz...") 
df_resultados_kfold_maiz_rf = kfold_cult_koppen(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_kfold_arroz_rf = kfold_cult_koppen(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_kfold_trigo_rf = kfold_cult_koppen(trigo, col_names_train_wheat, model_rf)
print("Modelos entrenados.")

Entrenamos Random Forest con K-Fold por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [27]:
print("Entrenamos XGBoost con K_Fold por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_xgb = kfold_cult_koppen(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_xgb = kfold_cult_koppen(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_xgb = kfold_cult_koppen(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con K_Fold por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [28]:
print("Entrenamos LightGBM con K-Fold por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_lgb = kfold_cult_koppen(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_lgb = kfold_cult_koppen(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_lgb = kfold_cult_koppen(trigo, col_names_train_wheat, model_lgb)

Entrenamos LightGBM con K-Fold por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...


In [29]:
print("Entrenamos SVR con K-Fold por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_svr = kfold_cult_koppen(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_kfold_arroz_svr = kfold_cult_koppen(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_kfold_trigo_svr = kfold_cult_koppen(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con K-Fold por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [30]:
df_resultados_kfold_maiz_rf.to_csv("./Resultados/resultados_kfold_maiz_rf2.csv", index=False)
df_resultados_kfold_arroz_rf.to_csv("./Resultados/resultados_kfold_arroz_rf2.csv", index=False)
df_resultados_kfold_trigo_rf.to_csv("./Resultados/resultados_kfold_trigo_rf2.csv", index=False)

df_resultados_kfold_maiz_xgb.to_csv("./Resultados/resultados_kfold_maiz_xgb2.csv", index=False)
df_resultados_kfold_arroz_xgb.to_csv("./Resultados/resultados_kfold_arroz_xgb2.csv", index=False)
df_resultados_kfold_trigo_xgb.to_csv("./Resultados/resultados_kfold_trigo_xgb2.csv", index=False)

df_resultados_kfold_maiz_lgb.to_csv("./Resultados/resultados_kfold_maiz_lgb2.csv", index=False)
df_resultados_kfold_arroz_lgb.to_csv("./Resultados/resultados_kfold_arroz_lgb2.csv", index=False)
df_resultados_kfold_trigo_lgb.to_csv("./Resultados/resultados_kfold_trigo_lgb2.csv", index=False)

df_resultados_kfold_maiz_svr.to_csv("./Resultados/resultados_kfold_maiz_svr2.csv", index=False)
df_resultados_kfold_arroz_svr.to_csv("./Resultados/resultados_kfold_arroz_svr2.csv", index=False)
df_resultados_kfold_trigo_svr.to_csv("./Resultados/resultados_kfold_trigo_svr2.csv", index=False)

##### Ventana Temporal

In [31]:
def vent_temp_cult_koppen(df_cultivo, col_names_cultivo, model):
    """
    Crea un DataFrame con los resultados de Ventana Temporal por cultivo y clase de Köppen.
    
    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados de Ventana Temporal por cultivo y clase de Köppen.
    """
    resultados = []

    # Lista de climas
    climas = df_cultivo['koppen_class'].unique().tolist()

    for clima in climas:
        df_clima = df_cultivo[df_cultivo["koppen_class"] == clima]
        df_clima = df_clima.sort_values(by="year")
        anyos = df_clima["year"].unique().tolist()

        if len(anyos) < 5:
            print(f"Menos de 5 años de datos para {clima}, saltando...")
            continue

        for i in range(3, len(anyos)):
            anyo = anyos[i]
            anyos_ant = anyos[:i] # Todos los años anteriores al actual                                                                         
            df_train = df_clima[df_clima["year"].isin(anyos_ant)]
            df_test = df_clima[df_clima["year"] == anyo] # Test
            koppen = df_test["koppen_class"].iloc[0]

            X_train = df_train[col_names_cultivo[:-1] + ["year"]]
            y_train = df_train["yield"]
            X_test = df_test[col_names_cultivo[:-1] + ["year"]]
            y_test = df_test["yield"]

            X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
            X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

            if len(X_train) == 0 or len(X_test) == 0:
                continue

            # Entrenamiento del modelo
            model_current = clone(model)  
            model_current.fit(X_train, y_train)

            # Predecimos
            y_pred = model_current.predict(X_test)
            
            # Evaluamos las predicciones del modelo
            # r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
            mae = mean_absolute_error(y_test, y_pred) # Error absoluto medio
            me = np.sum(y_test - y_pred) / len(y_test) # Error medio

            # Normalizamos los resultados
            nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
            rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
            mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
            nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

            resultados.append({
                "koppen_class": koppen,
                "year": anyo,
                "y_true": y_test.values.tolist(),
                "y_pred": y_pred.tolist(),
                # "R2": r2 if r2 is not None else 0,
                "NRMSE": nrmse,
                "rRMSE": rRMSE,
                "NMAE": mae,
                "nME": nme,
            })

    return pd.DataFrame(resultados)

In [32]:
print("Entrenando modelos con Ventana Temporal por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_rf = vent_temp_cult_koppen(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_rf = vent_temp_cult_koppen(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_rf = vent_temp_cult_koppen(trigo, col_names_train_wheat, model_rf)
print("Modelos entrenados.")

Entrenando modelos con Ventana Temporal por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [33]:
print("Entrenamos XGBoost con ventana temporal por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_xgb = vent_temp_cult_koppen(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_xgb = vent_temp_cult_koppen(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_xgb = vent_temp_cult_koppen(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con ventana temporal por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [34]:
print("Entrenamos LightGBM con ventana temporal por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_lgb = vent_temp_cult_koppen(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_lgb = vent_temp_cult_koppen(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_lgb = vent_temp_cult_koppen(trigo, col_names_train_wheat, model_lgb)
print("Modelos entrenados.")

Entrenamos LightGBM con ventana temporal por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [35]:
print("Entrenamos SVR con ventana temporal por clase de Köppen...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_svr = vent_temp_cult_koppen(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_svr = vent_temp_cult_koppen(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_svr = vent_temp_cult_koppen(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con ventana temporal por clase de Köppen...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [36]:
df_resultados_vent_temp_maiz_rf.to_csv("./Resultados/resultados_vent_temp_maiz_rf2.csv", index=False)
df_resultados_vent_temp_arroz_rf.to_csv("./Resultados/resultados_vent_temp_arroz_rf2.csv", index=False)
df_resultados_vent_temp_trigo_rf.to_csv("./Resultados/resultados_vent_temp_trigo_rf2.csv", index=False)

df_resultados_vent_temp_maiz_xgb.to_csv("./Resultados/resultados_vent_temp_maiz_xgb2.csv", index=False)
df_resultados_vent_temp_arroz_xgb.to_csv("./Resultados/resultados_vent_temp_arroz_xgb2.csv", index=False)
df_resultados_vent_temp_trigo_xgb.to_csv("./Resultados/resultados_vent_temp_trigo_xgb2.csv", index=False)

df_resultados_vent_temp_maiz_lgb.to_csv("./Resultados/resultados_vent_temp_maiz_lgb2.csv", index=False)
df_resultados_vent_temp_arroz_lgb.to_csv("./Resultados/resultados_vent_temp_arroz_lgb2.csv", index=False)
df_resultados_vent_temp_trigo_lgb.to_csv("./Resultados/resultados_vent_temp_trigo_lgb2.csv", index=False)

df_resultados_vent_temp_maiz_svr.to_csv("./Resultados/resultados_vent_temp_maiz_svr2.csv", index=False)
df_resultados_vent_temp_arroz_svr.to_csv("./Resultados/resultados_vent_temp_arroz_svr2.csv", index=False)
df_resultados_vent_temp_trigo_svr.to_csv("./Resultados/resultados_vent_temp_trigo_svr2.csv", index=False)

#### 3. Entrenamiento de modelos agrupando por cultivo

Procedimiento: Analizar, para cada cultivo, los datos en base a cada una de las estrategias de entrenamiento.

##### K-Fold

In [37]:
def kfold_cultivo(df_cultivo, col_names_cultivo, model):
    """
    Entrena un modelo para cada cultivo usando K-Fold por año.

    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados de K-Fold por cultivo.
    """

    resultados = []

    anyos = df_cultivo["year"].unique().tolist()

    if len(anyos) < 5:
        print(f"Menos de 5 años de datos, saltando...")
        return pd.DataFrame(resultados)
    
    for anyo in anyos:
        df_train = df_cultivo[df_cultivo["year"] != anyo]
        df_test = df_cultivo[df_cultivo["year"] == anyo]

        X_train = df_train[col_names_cultivo[:-1] + ["year"]]
        y_train = df_train["yield"]
        X_test = df_test[col_names_cultivo[:-1] + ["year"]]
        y_test = df_test["yield"]

        X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
        X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

        # Verificamos si hay suficientes datos para entrenar
        if len(X_train) == 0 or len(X_test) ==0:
            print(f"Datos insuficientes para entrenar el modelo en el año {anyo}, saltando...")
            continue

        # Entrenamiento del modelo
        model_current = clone(model)
        model_current.fit(X_train, y_train)
        
        # Predecimos
        y_pred = model_current.predict(X_test)

        # Evaluamos las predicciones del modelo
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
        mae = mean_absolute_error(y_test, y_pred) # Error absoluto medio
        me = np.sum(y_test - y_pred) / len(y_test) # Error medio

        # Normalizamos los resultados
        nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
        rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
        mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
        nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

        resultados.append({
            "year": anyo,
            "y_true": y_test.values.tolist(),
            "y_pred": y_pred.tolist(),
            "R2": r2,
            "NRMSE": nrmse,
            "rRMSE": rRMSE,
            "NMAE": mae,
            "nME": nme,
        })

    df_resultados = pd.DataFrame(resultados)

    # Entrenamos un modelo final sobre todos los datos
    X_total = df_cultivo[col_names_cultivo[:-1] + ["year"]]
    y_total = df_cultivo["yield"]
    X_total.columns = X_total.columns.str.replace(":", "_", regex=False)

    model_final = clone(model)
    model_final.fit(X_total, y_total)


    return df_resultados, model_final

In [38]:
print("Entrenamos Random Forest con K-Fold")
print("Entrenando maíz...")
df_resultados_kfold_maiz_rf, aux = kfold_cultivo(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_kfold_arroz_rf, aux = kfold_cultivo(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_kfold_trigo_rf, aux = kfold_cultivo(trigo, col_names_train_wheat, model_rf)

Entrenamos Random Forest con K-Fold
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...


In [39]:
print("Entrenamos XGBoost con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_xgb, aux = kfold_cultivo(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_xgb, aux = kfold_cultivo(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_xgb, aux = kfold_cultivo(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [40]:
print("Entrenamos LightGBM con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_lgb, aux = kfold_cultivo(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_kfold_arroz_lgb, aux = kfold_cultivo(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_kfold_trigo_lgb, aux = kfold_cultivo(trigo, col_names_train_wheat, model_lgb)
print("Modelos entrenados.")

Entrenamos LightGBM con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [41]:
print("Entrenamos SVR con K-Fold...")
print("Entrenando maíz...")
df_resultados_kfold_maiz_svr, aux = kfold_cultivo(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_kfold_arroz_svr, aux = kfold_cultivo(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_kfold_trigo_svr, modelo_svr_kfold_trigo3 = kfold_cultivo(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con K-Fold...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [42]:
df_resultados_kfold_maiz_rf.to_csv("./Resultados/resultados_kfold_maiz_rf3.csv", index=False)
df_resultados_kfold_arroz_rf.to_csv("./Resultados/resultados_kfold_arroz_rf3.csv", index=False)
df_resultados_kfold_trigo_rf.to_csv("./Resultados/resultados_kfold_trigo_rf3.csv", index=False)

df_resultados_kfold_maiz_xgb.to_csv("./Resultados/resultados_kfold_maiz_xgb3.csv", index=False)
df_resultados_kfold_arroz_xgb.to_csv("./Resultados/resultados_kfold_arroz_xgb3.csv", index=False)
df_resultados_kfold_trigo_xgb.to_csv("./Resultados/resultados_kfold_trigo_xgb3.csv", index=False)

df_resultados_kfold_maiz_lgb.to_csv("./Resultados/resultados_kfold_maiz_lgb3.csv", index=False)
df_resultados_kfold_arroz_lgb.to_csv("./Resultados/resultados_kfold_arroz_lgb3.csv", index=False)
df_resultados_kfold_trigo_lgb.to_csv("./Resultados/resultados_kfold_trigo_lgb3.csv", index=False)

df_resultados_kfold_maiz_svr.to_csv("./Resultados/resultados_kfold_maiz_svr3.csv", index=False)
df_resultados_kfold_arroz_svr.to_csv("./Resultados/resultados_kfold_arroz_svr3.csv", index=False)
df_resultados_kfold_trigo_svr.to_csv("./Resultados/resultados_kfold_trigo_svr3.csv", index=False)
joblib.dump(modelo_svr_kfold_trigo3, "./Mejores_modelos/training_kfold_trigo_svr3.pkl")

['./Mejores_modelos/training_kfold_trigo_svr3.pkl']

##### Ventana Temporal

In [43]:
def vent_temp_cultivo(df_cultivo, col_names_cultivo, model):
    """
    Entrena un modelo usando una ventana temporal.

    Parámetros:
    df_cultivo (DataFrame): DataFrame con los datos del cultivo.
    col_names_cultivo (list): Lista de nombres de columnas para el entrenamiento.

    Retorna:
    DataFrame: DataFrame con los resultados del modelo.
    """
    
    resultados = []

    anyos = df_cultivo["year"].unique().tolist()

    if len(anyos) < 5:
        print(f"Menos de 5 años de datos, saltando...")
        return pd.DataFrame(resultados)
    
    for i in range(3, len(anyos)):
        anyo = anyos[i]
        anyos_ant = anyos[:i] # Todos los años anteriores al actual

        df_train = df_cultivo[df_cultivo["year"].isin(anyos_ant)]
        df_test = df_cultivo[df_cultivo["year"] == anyo]

        X_train = df_train[col_names_cultivo[:-1] + ["year"]]
        y_train = df_train["yield"]
        X_test = df_test[col_names_cultivo[:-1] + ["year"]]
        y_test = df_test["yield"]

        X_train.columns = X_train.columns.str.replace(":", "_", regex=False)
        X_test.columns = X_test.columns.str.replace(":", "_", regex=False)

        if len(X_train) == 0 or len(X_test) == 0:
            continue

        # Entrenamiento del modelo
        model_current = clone(model)
        model_current.fit(X_train, y_train)
        
        # Predecimos
        y_pred = model_current.predict(X_test)

        # Evaluamos las predicciones del modelo
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
        mae = mean_absolute_error(y_test, y_pred) # Error absoluto medio
        me = np.sum(y_test - y_pred) / len(y_test) # Error medio

        # Normalizamos los resultados
        nrmse = rmse / np.mean(y_test) if np.mean(y_test) != 0 else rmse
        rRMSE = (rmse / np.mean(y_test))*100 # Error cuadrático medio relativo
        mae = mae / np.mean(y_test) if np.mean(y_test) != 0 else mae
        nme = me / np.mean(y_test) if np.mean(y_test) != 0 else me

        resultados.append({
            "year": anyo,
            "y_true": y_test.values.tolist(),
            "y_pred": y_pred.tolist(),
            "R2": r2,
            "NRMSE": nrmse,
            "rRMSE": rRMSE,
            "NMAE": mae,
            "nME": nme,
        })

    return pd.DataFrame(resultados)

In [44]:
print("Entrenamos Random Forest con Ventana Temporal")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_rf = vent_temp_cultivo(maiz, col_names_train_maize, model_rf)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_rf = vent_temp_cultivo(arroz, col_names_train_rice, model_rf)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_rf = vent_temp_cultivo(trigo, col_names_train_wheat, model_rf)
print("Modelos entrenados.")

Entrenamos Random Forest con Ventana Temporal
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [45]:
print("Entrenamos XGBoost con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_xgb = vent_temp_cultivo(maiz, col_names_train_maize, model_xgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_xgb = vent_temp_cultivo(arroz, col_names_train_rice, model_xgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_xgb = vent_temp_cultivo(trigo, col_names_train_wheat, model_xgb)
print("Modelos entrenados.")

Entrenamos XGBoost con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [46]:
print("Entrenamos LightGBM con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_lgb = vent_temp_cultivo(maiz, col_names_train_maize, model_lgb)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_lgb = vent_temp_cultivo(arroz, col_names_train_rice, model_lgb)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_lgb = vent_temp_cultivo(trigo, col_names_train_wheat, model_lgb)
print("Modelos entrenados.")

Entrenamos LightGBM con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [47]:
print("Entrenamos SVR con ventana temporal...")
print("Entrenando maíz...")
df_resultados_vent_temp_maiz_svr = vent_temp_cultivo(maiz, col_names_train_maize, params_svr_maiz_final)
print("Entrenando arroz...")
df_resultados_vent_temp_arroz_svr = vent_temp_cultivo(arroz, col_names_train_rice, params_svr_arroz_final)
print("Entrenando trigo...")
df_resultados_vent_temp_trigo_svr = vent_temp_cultivo(trigo, col_names_train_wheat, params_svr_trigo_final)
print("Modelos entrenados.")

Entrenamos SVR con ventana temporal...
Entrenando maíz...
Entrenando arroz...
Entrenando trigo...
Modelos entrenados.


In [48]:
df_resultados_vent_temp_maiz_rf.to_csv("./Resultados/resultados_vent_temp_maiz_rf3.csv", index=False)
df_resultados_vent_temp_arroz_rf.to_csv("./Resultados/resultados_vent_temp_arroz_rf3.csv", index=False)
df_resultados_vent_temp_trigo_rf.to_csv("./Resultados/resultados_vent_temp_trigo_rf3.csv", index=False)

df_resultados_vent_temp_maiz_xgb.to_csv("./Resultados/resultados_vent_temp_maiz_xgb3.csv", index=False)
df_resultados_vent_temp_arroz_xgb.to_csv("./Resultados/resultados_vent_temp_arroz_xgb3.csv", index=False)
df_resultados_vent_temp_trigo_xgb.to_csv("./Resultados/resultados_vent_temp_trigo_xgb3.csv", index=False)

df_resultados_vent_temp_maiz_lgb.to_csv("./Resultados/resultados_vent_temp_maiz_lgb3.csv", index=False)
df_resultados_vent_temp_arroz_lgb.to_csv("./Resultados/resultados_vent_temp_arroz_lgb3.csv", index=False)
df_resultados_vent_temp_trigo_lgb.to_csv("./Resultados/resultados_vent_temp_trigo_lgb3.csv", index=False)

df_resultados_vent_temp_maiz_svr.to_csv("./Resultados/resultados_vent_temp_maiz_svr3.csv", index=False)
df_resultados_vent_temp_arroz_svr.to_csv("./Resultados/resultados_vent_temp_arroz_svr3.csv", index=False)
df_resultados_vent_temp_trigo_svr.to_csv("./Resultados/resultados_vent_temp_trigo_svr3.csv", index=False) 